In [0]:
from pyspark.sql import functions as f

In [0]:
from pyspark.sql import functions as f

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS catalogmeteo.gold;

### CALCULATION OF AVERAGE TEMPERATURE, AVERAGE WIND AND AVERAGE PRESSURE PER AIRPORT

In [0]:


# 1. Load DataFrames
obs_df = spark.table("catalogmeteo.silver.weather_observations") \
    .withColumn("period", f.date_format(f.col("date"), "yyyy-MM"))

meta_df = spark.table("catalogmeteo.silver.weather_stations")

# 2. Join using specific DataFrame references to avoid ambiguity
# We join on station_id, which Spark handles automatically if it's the only common column
joined_df = meta_df.join(obs_df, "station_id", "inner")

# 3. Pivot with explicit column references
weather_pivot = joined_df.groupBy(
    meta_df.station_id,    # Explicitly use meta_df version
    meta_df.station_name, 
    meta_df.city,          # This fixes the [AMBIGUOUS_REFERENCE] error
    meta_df.elevation_m
).pivot("period").agg(
    f.round(f.avg("temperature_c"), 2).alias("avg_temp"),
    f.round(f.avg("humidity_pct"), 2).alias("avg_humidity"),
    f.round(f.avg("wind_speed_kmh"), 2).alias("avg_wind_speed")
)

In [0]:
weather_pivot.write.format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("catalogmeteo.gold.weather_kpi")

### ottimizzazione managed table del catalog

In [0]:
%sql
OPTIMIZE catalogmeteo.gold.weather_kpi ZORDER BY station_id;

In [0]:
%sql
VACUUM catalogmeteo.gold.weather_kpi;

### salva la tabella (external table) in gold di adls

In [0]:
# Write to a specific ADLS container
weather_pivot.write.format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .save("abfss://gold@storageaccountmeteo.dfs.core.windows.net/tables/weather_kpi")

### ottimizzazione external table di adls

In [0]:
%sql
OPTIMIZE "abfss://gold@storageaccountmeteo.dfs.core.windows.net/tables/weather_kpi" ZORDER BY station_id;

In [0]:
%sql
VACUUM "abfss://gold@storageaccountmeteo.dfs.core.windows.net/tables/weather_kpi";